# Introduction to Convolutional Neural Networks and AlexNet

In this notebook, we will:
- Explore key building blocks of CNNs.
- Understand how different types of layers work (Convolution, Padding, BatchNorm, LayerNorm, Transposed Convolution, Pooling).
- Build AlexNet layer-by-layer in PyTorch.
- Load pretrained AlexNet weights from the internet and run a forward inference.

Let's get started!


## Understanding Different Types of Layers

### 1. Convolutional Layers
- **Purpose:** Extract spatial features from images using learnable filters.
- **Key Parameters:** Number of input/output channels, kernel size, stride, and padding.
  


In [1]:
# Let's demonstrate a simple convolutional layer.
import torch
import torch.nn as nn

# Create a single convolutional layer: 3 input channels, 16 output channels, 3x3 kernel.
conv_layer = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=1)
dummy_input = torch.randn(1, 3, 32, 32)  # Example: batch of 1, 32x32 image
conv_output = conv_layer(dummy_input)
print("Convolution output shape:", conv_output.shape)


Convolution output shape: torch.Size([1, 16, 32, 32])


### 2. Padding
- **Purpose:** Control the spatial size of the output by adding borders to the input.
- **Types:** Zero-padding is most common.


In [2]:
# Demonstrate padding using nn.ZeroPad2d and how it affects output size.
import torch.nn.functional as F

# Apply 2 pixels of zero padding on all sides.
pad = nn.ZeroPad2d(2)
padded_input = pad(dummy_input)
print("Original shape:", dummy_input.shape)
print("Padded shape:", padded_input.shape)


Original shape: torch.Size([1, 3, 32, 32])
Padded shape: torch.Size([1, 3, 36, 36])



### 3. Batch Normalization
- **Purpose:** Normalize the inputs of each mini-batch, which stabilizes and speeds up training.
  

In [3]:
# Demonstrate Batch Normalization and Layer Normalization.
# BatchNorm normalizes across the batch and spatial dimensions per channel.
batch_norm = nn.BatchNorm2d(16)  # for 16 channels as from the previous conv output
bn_output = batch_norm(conv_output)
print("BatchNorm output shape:", bn_output.shape)

# LayerNorm normalizes across all features in each sample.
# For a convolutional output, you may want to normalize over the [C, H, W] dimensions.
layer_norm = nn.LayerNorm(conv_output.shape[1:])
ln_output = layer_norm(conv_output[0])  # apply on one sample
print("LayerNorm output shape (single sample):", ln_output.shape)


BatchNorm output shape: torch.Size([1, 16, 32, 32])
LayerNorm output shape (single sample): torch.Size([16, 32, 32])


### 4. Layer Normalization
- **Purpose:** Similar to BatchNorm but normalizes across the features of each sample independently, useful in some settings where batch statistics are less reliable.
  

In [4]:
# Demonstrate a Transposed Convolution (often used for upsampling).
# Here, we reverse the spatial reduction of a normal convolution.
trans_conv = nn.ConvTranspose2d(in_channels=16, out_channels=8, kernel_size=3, stride=2, padding=1, output_padding=1)
trans_conv_output = trans_conv(conv_output)
print("Transposed Convolution output shape:", trans_conv_output.shape)


Transposed Convolution output shape: torch.Size([1, 8, 64, 64])



### 5. Transposed Convolution (Deconvolution)
- **Purpose:** Increase the spatial dimensions (width and height) of the feature maps.
- **Use Case:** Commonly used in generative networks or upsampling tasks.
  

In [5]:
# Demonstrate Pooling layers.
# Max pooling
max_pool = nn.MaxPool2d(kernel_size=2, stride=2)
max_pool_output = max_pool(conv_output)
print("MaxPool output shape:", max_pool_output.shape)


MaxPool output shape: torch.Size([1, 16, 16, 16])




### 6. Pooling Layers
- **Purpose:** Downsample feature maps by summarizing regions (e.g., via max or average operations) to reduce spatial dimensions and computational load.


In [6]:
# Average pooling
avg_pool = nn.AvgPool2d(kernel_size=2, stride=2)
avg_pool_output = avg_pool(conv_output)
print("AvgPool output shape:", avg_pool_output.shape)


AvgPool output shape: torch.Size([1, 16, 16, 16])


## Building AlexNet from Scratch

![AlexNet](alexNet-architecture.png)

We now build the AlexNet architecture. Recall the architecture includes:
- **5 Convolutional Layers:** with ReLU activations and some max pooling.
- **Fully Connected Layers:** after flattening the features.
- **Dropout:** to reduce overfitting.

In our implementation:
- The `features` block will handle convolution and pooling.
- The `classifier` block will flatten the output and apply fully connected layers.



In [7]:
import torch.nn as nn

class AlexNet(nn.Module):
    def __init__(self, num_classes=1000):
        super(AlexNet, self).__init__()
        self.features = nn.Sequential(
            # Conv Layer 1: 11x11 kernel, stride 4
            nn.Conv2d(3, 96, kernel_size=11, stride=4, padding=0),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
            
            # Conv Layer 2: 5x5 kernel with padding
            nn.Conv2d(96, 256, kernel_size=5, stride=1, padding=2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
            
            # Conv Layer 3: 3x3 kernel
            nn.Conv2d(256, 384, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            
            # Conv Layer 4: 3x3 kernel
            nn.Conv2d(384, 384, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            
            # Conv Layer 5: 3x3 kernel
            nn.Conv2d(384, 256, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2)
        )
        
        self.classifier = nn.Sequential(
            nn.Dropout(),
            nn.Linear(256 * 6 * 6, 4096),  # Adjusted for input image size 224x224
            nn.ReLU(inplace=True),
            nn.Dropout(),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            nn.Linear(4096, num_classes)
        )
        
    def forward(self, x):
        x = self.features(x)
        # Flatten the output for the fully connected layers
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

# Instantiate our AlexNet model for 10 classes (for example, if working with CIFAR-10)
model = AlexNet(num_classes=10)
print(model)


AlexNet(
  (features): Sequential(
    (0): Conv2d(3, 96, kernel_size=(11, 11), stride=(4, 4))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(96, 256, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(256, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): Conv2d(384, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU(inplace=True)
    (10): Conv2d(384, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Dropout(p=0.5, inplace=False)
    (1): Linear(in_features=9216, out_features=4096, bias=True)
    (2): ReLU(inplace=True)
    (3): Dropout(p=0.5, inplace=False)
 

## Loading Pretrained AlexNet and Running Inference

PyTorch's `torchvision.models` provides a pretrained AlexNet (trained on ImageNet). In the next cell, we:
- Load the pretrained model.
- Prepare a dummy input image (or you can load a real image with proper transforms).
- Run a forward pass to see the model predictions.

![Dog](dog.png)

In [8]:
import torch
from torchvision import models, transforms
from PIL import Image
import requests
from io import BytesIO

# Load the pretrained AlexNet model
pretrained_alexnet = models.alexnet(pretrained=True)
pretrained_alexnet.eval()  # Set model to evaluation mode

# Define a transform pipeline to resize, center crop, and normalize the image as expected by AlexNet
transform_pipeline = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                         std=[0.229, 0.224, 0.225])
])

# Open the image
img = Image.open('dog.png').convert("RGB")

# Apply the transformations and add a batch dimension
img_t = transform_pipeline(img).unsqueeze(0)

# Run inference
with torch.no_grad():
    output = pretrained_alexnet(img_t)

# Get the top 5 predicted indices
_, top5 = torch.topk(output, 5)

# Download ImageNet class names
labels_url = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"
response = requests.get(labels_url)
imagenet_classes = response.text.splitlines()

# Print the top 5 predicted class names along with their indices
print("Top 5 predicted classes:")
for idx in top5[0].tolist():
    print(f"Index: {idx}, Class: {imagenet_classes[idx]}")


/home/loki/miniconda3/envs/pz/lib/python3.8/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/loki/miniconda3/envs/pz/lib/python3.8/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Top 5 predicted classes:
Index: 207, Class: golden retriever
Index: 219, Class: cocker spaniel
Index: 220, Class: Sussex spaniel
Index: 208, Class: Labrador retriever
Index: 852, Class: tennis ball


## Wrap-Up and Further Exploration

In this notebook, we:
- Reviewed key CNN components and how layers such as convolution, normalization, and pooling work.
- Built the AlexNet architecture layer-by-layer.
- Loaded a pretrained AlexNet model and demonstrated a forward pass on a sample image.

**Next Steps:**
- Experiment with modifying the AlexNet architecture.
- Explore training and fine-tuning on your own datasets.
- Investigate how different layers impact model performance and training dynamics.

Feel free to ask questions or experiment further!
